# Mapping Process Showcase Run
Goal: present a clear, step-by-step lineage from cleaned inputs to semantic mapping outputs and program-level results.

Expected outputs:
- Input data overview tables
- Cleaning-to-mapping lineage samples
- Candidate mapping summary metrics
- Program-level final table preview

## 1) Create a New Notebook Run Goal
This notebook is a presentation-ready walkthrough of the mapping pipeline.

Flow shown in this run:
1. Load cleaned inputs and mapping outputs.
2. Show how data moves from cleaned fields to mapped candidates.
3. Reconstruct a traceable program-level view.
4. Validate that outputs are consistent for reporting.

This is a baseline integration run before GCSI scoring.

## 2) Set Up Environment and Imports
This cell imports libraries needed for table operations and display.
- `pandas`, `numpy`: data handling
- `pathlib`: portable file paths
- `IPython.display`: cleaner notebook display

Note: this run reads already-generated mapping outputs, so it does not require rerunning Sentence Transformer inference.

In [1]:
# Core libraries for showcase run
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Environment setup complete.")

Environment setup complete.


## 3) Define Run Configuration
All file paths and runtime parameters are grouped here so the run is easy to repeat.

In [8]:
# Centralized run configuration
CONFIG = {
    "base_dir": Path("."),
    "masters_clean_path": Path("datasets/cleaned/masters_cleaned_for_mapping.csv"),
    "occupations_clean_path": Path("datasets/cleaned/occupations_cleaned_for_mapping.csv"),
    "degree_candidates_path": Path("datasets/mapping/degree_career_sentence_transformer_candidates_thr30.csv"),
    "degree_candidate_stats_path": Path("datasets/mapping/degree_career_sentence_transformer_candidate_stats_thr30.csv"),
    "program_candidates_long_path": Path("datasets/mapping/program_career_sentence_transformer_candidates_thr30_long.csv"),
    "program_best_path": Path("datasets/mapping/program_career_sentence_transformer_best_thr30.csv"),
    "show_rows": 10,
    "analysis_threshold": 0.30,
    "sample_size": 5,
    "sample_seed": 42,
}

for k, v in CONFIG.items():
    if "path" in k:
        print(f"{k}: {v}")

masters_clean_path: datasets/cleaned/masters_cleaned_for_mapping.csv
occupations_clean_path: datasets/cleaned/occupations_cleaned_for_mapping.csv
degree_candidates_path: datasets/mapping/degree_career_sentence_transformer_candidates_thr30.csv
degree_candidate_stats_path: datasets/mapping/degree_career_sentence_transformer_candidate_stats_thr30.csv
program_candidates_long_path: datasets/mapping/program_career_sentence_transformer_candidates_thr30_long.csv
program_best_path: datasets/mapping/program_career_sentence_transformer_best_thr30.csv


## 4) Load or Paste the Target Code
These helper functions load data, generate lineage samples, and prepare summary outputs for the showcase.

In [9]:
def assert_files_exist(cfg):
    required = [
        "masters_clean_path",
        "occupations_clean_path",
        "degree_candidates_path",
        "degree_candidate_stats_path",
        "program_candidates_long_path",
        "program_best_path",
    ]
    missing = [k for k in required if not cfg[k].exists()]
    if missing:
        missing_paths = {k: str(cfg[k]) for k in missing}
        raise FileNotFoundError(f"Missing required files: {missing_paths}")


def load_showcase_data(cfg):
    masters = pd.read_csv(cfg["masters_clean_path"])
    occupations = pd.read_csv(cfg["occupations_clean_path"])
    degree_candidates = pd.read_csv(cfg["degree_candidates_path"])
    degree_stats = pd.read_csv(cfg["degree_candidate_stats_path"])
    program_long = pd.read_csv(cfg["program_candidates_long_path"])
    program_best = pd.read_csv(cfg["program_best_path"])
    return {
        "masters": masters,
        "occupations": occupations,
        "degree_candidates": degree_candidates,
        "degree_stats": degree_stats,
        "program_long": program_long,
        "program_best": program_best,
    }


def build_five_program_trace(data, cfg):
    masters = data["masters"].copy().reset_index(drop=True)
    masters["program_row_id"] = masters.index + 1

    # Deterministic 5-program sample from mapped programs
    sample_programs = (
        data["program_best"][["program_row_id", "program_name", "program_type", "degree_source", "degree_final"]]
        .drop_duplicates()
        .sample(n=cfg["sample_size"], random_state=cfg["sample_seed"])
        .sort_values("program_row_id")
        .reset_index(drop=True)
    )

    lineage_cols = ["program_row_id", "program_name", "program_type"]
    for col in ["program_name_clean", "degree_text", "degree_norm"]:
        if col in masters.columns:
            lineage_cols.append(col)

    cleaning_lineage = sample_programs[["program_row_id", "degree_source", "degree_final"]].merge(
        masters[lineage_cols],
        on="program_row_id",
        how="left",
    )

    # Keep top candidate rows per sample program for concise display
    sample_mapping = (
        data["program_long"]
        .merge(sample_programs[["program_row_id"]], on="program_row_id", how="inner")
        .sort_values(["program_row_id", "match_rank"])
        .groupby("program_row_id", as_index=False, group_keys=False)
        .head(8)
        .reset_index(drop=True)
    )

    return sample_programs, cleaning_lineage, sample_mapping


def summarize_run(data):
    masters = data["masters"]
    occupations = data["occupations"]
    degree_candidates = data["degree_candidates"]
    degree_stats = data["degree_stats"]
    program_long = data["program_long"]
    program_best = data["program_best"]

    summary = {
        "masters_rows": len(masters),
        "occupations_rows": len(occupations),
        "unique_degree_labels": degree_candidates["degree_clean"].nunique(),
        "unique_occupation_labels": degree_candidates["career_match"].nunique(),
        "degree_candidate_rows": len(degree_candidates),
        "program_candidate_rows": len(program_long),
        "program_best_rows": len(program_best),
        "avg_candidates_per_degree": round(float(degree_stats["candidate_count"].mean()), 2),
        "median_candidates_per_degree": int(degree_stats["candidate_count"].median()),
        "similarity_min": round(float(degree_candidates["similarity"].min()), 4),
        "similarity_max": round(float(degree_candidates["similarity"].max()), 4),
    }
    return pd.DataFrame([summary])


def run_showcase(cfg):
    assert_files_exist(cfg)
    data = load_showcase_data(cfg)
    sample_programs, cleaning_lineage, sample_mapping = build_five_program_trace(data, cfg)
    summary_df = summarize_run(data)

    return {
        "data": data,
        "summary": summary_df,
        "sample_programs": sample_programs,
        "cleaning_lineage": cleaning_lineage,
        "sample_mapping": sample_mapping,
    }

## 5) Execute the Run Cell
This cell runs the showcase pipeline and prints key progress markers.

In [10]:
print("[1/3] Running showcase pipeline...")
showcase = run_showcase(CONFIG)

print("[2/3] Loaded dataframes:")
for name, df in showcase["data"].items():
    print(f"  - {name}: {df.shape}")

print("[3/3] Showcase run complete.")

[1/3] Running showcase pipeline...
[2/3] Loaded dataframes:
  - masters: (57085, 6)
  - occupations: (1120, 3)
  - degree_candidates: (1215700, 4)
  - degree_stats: (22641, 4)
  - program_long: (4357770, 8)
  - program_best: (57085, 8)
[3/3] Showcase run complete.


## 6) Display and Explain Results
These cells show the end-to-end lineage and key metrics to explain what happened in the run.

In [11]:
print("Run summary metrics")
display(showcase["summary"])

print("\nInput preview: cleaned masters")
display(showcase["data"]["masters"].head(CONFIG["show_rows"]))

print("\nInput preview: cleaned occupations")
display(showcase["data"]["occupations"].head(CONFIG["show_rows"]))

print("\nFixed 5-sample programs used throughout")
display(showcase["sample_programs"])

print("\nCleaning lineage for the same 5 samples")
display(showcase["cleaning_lineage"])

print("\nMapping candidates for the same 5 samples")
display(showcase["sample_mapping"])

Run summary metrics


,masters_rows,occupations_rows,unique_degree_labels,unique_occupation_labels,degree_candidate_rows,program_candidate_rows,program_best_rows,avg_candidates_per_degree,median_candidates_per_degree,similarity_min,similarity_max
0,57085,1120,22641,1119,1215700,4357770,57085,53.69,35,0.3,1.0



Input preview: cleaned masters


,program_name,program_type,program_name_clean,degree_text,degree_norm,remove_flag
0,Economics,MSc,economics,economics,economics,False
1,Political Science and International Affairs,Master,political science and international affairs,political science and international affairs,political science and international affairs,False
2,Business Administration,MBA,business administration,business administration,business administration,False
3,Computer and Information Science,MSc,computer and information science,computer and information science,computer science,False
4,Industrial Engineering and Systems Management,MEng,industrial engineering and systems management,industrial engineering and systems management,industrial engineering and systems management,False
5,Law,LLM,law,law,law,False
6,Public Health Program,Master,public health program,public health,public health,False
7,Master of Business Administration in Tourism a...,MBA,master of business administration in tourism a...,business administration in tourism and interna...,business administration in tourism and interna...,False
8,Aruban Law,LLM,aruban law,aruban law,aruban law,False
9,Human Resource Management,Master,human resource management,human resource management,human resource management,False



Input preview: cleaned occupations


...

,program_row_id,program_name,program_type,degree_source,degree_final
0,5395,Social Science in Service Management,MSc,social science in service management,social science in service management
1,18154,Neurosciences,MSc,neurosciences,neurosciences
2,23972,Cardiovascular Health and Rehabilitation,MSc,cardiovascular health and rehabilitation,cardiovascular health and rehabilitation
3,31115,Environmental Management,MSc,environmental management,environmental management
4,35680,Earth Science,Master,earth science,earth science



Cleaning lineage for the same 5 samples


,program_row_id,degree_source,degree_final,program_name,program_type,program_name_clean,degree_text,degree_norm
0,5395,social science in service management,social science in service management,Social Science in Service Management,MSc,social science in service management,social science in service management,social science in service management
1,18154,neurosciences,neurosciences,Neurosciences,MSc,neurosciences,neurosciences,neurosciences
2,23972,cardiovascular health and rehabilitation,cardiovascular health and rehabilitation,Cardiovascular Health and Rehabilitation,MSc,cardiovascular health and rehabilitation,cardiovascular health and rehabilitation,cardiovascular health and rehabilitation
3,31115,environmental management,environmental management,Environmental Management,MSc,environmental management,environmental management,environmental management
4,35680,earth science,earth science,Earth Science,Master,earth science,earth science,earth science



Mapping candidates for the same 5 samples


,program_row_id,program_name,program_type,degree_source,degree_final,match_rank,career_match,similarity
0,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,1.0,social and community service managers,0.7400
1,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,2.0,community and social service occupations,0.6171
2,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,3.0,community and social service specialists all o...,0.6113
3,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,4.0,miscellaneous community and social service spe...,0.5759
4,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,5.0,social and human service assistants,0.5752
5,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,6.0,social science research assistants,0.5743
6,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,7.0,social scientists and related workers,0.5440
7,5395,Social Science in Service Management,MSc,social science in service management,social science in service management,8.0,social workers,0.5414
8,18154,Neurosciences,MSc,neurosciences,neurosciences,1.0,psychologists,0.6035
9,18154,Neurosciences,MSc,neurosciences,neurosciences,2.0,psychologists all other,0.5760


## 7) Add Validation Checks
These checks confirm that required columns exist, row counts are sensible, and threshold logic is respected.

In [12]:
data = showcase["data"]
summary = showcase["summary"].iloc[0]

required_cols_program = {"program_name", "program_type", "career_match", "similarity"}
assert required_cols_program.issubset(set(data["program_long"].columns)), "Program long table is missing required columns."

assert summary["masters_rows"] > 0, "Masters dataset is empty."
assert summary["occupations_rows"] > 0, "Occupations dataset is empty."
assert summary["degree_candidate_rows"] > 0, "No candidate mappings were generated."
assert summary["similarity_min"] >= CONFIG["analysis_threshold"], "Found similarity below configured threshold."

assert len(showcase["sample_programs"]) == CONFIG["sample_size"], "Sample size mismatch for fixed program trace."
assert showcase["sample_mapping"]["program_row_id"].nunique() == CONFIG["sample_size"], "Not all fixed sample programs reached mapping stage."

print("All validation checks passed.")

All validation checks passed.


## 8) Parameter Tweaks and Re-run
This comparison shows how candidate volume changes if we tighten the similarity threshold on the same candidate table.

In [7]:
candidate_df = showcase["data"]["degree_candidates"]

thresholds = [0.30, 0.35, 0.40, 0.45]
rows = []
for thr in thresholds:
    filtered = candidate_df[candidate_df["similarity"] >= thr]
    rows.append({
        "threshold": thr,
        "candidate_rows": len(filtered),
        "unique_degrees": filtered["degree_clean"].nunique(),
        "unique_careers": filtered["career_match"].nunique(),
        "avg_candidates_per_degree": round(
            float(filtered.groupby("degree_clean")["career_match"].count().mean()), 2
        ) if len(filtered) else 0.0,
    })

threshold_comparison = pd.DataFrame(rows)
display(threshold_comparison)

print("Interpretation: higher threshold reduces candidate volume and increases strictness.")

,threshold,candidate_rows,unique_degrees,unique_careers,avg_candidates_per_degree
0,0.30,1215700,22641,1119,53.69
1,0.35,639714,21912,1112,29.19
2,0.40,331462,20396,1091,16.25
3,0.45,169809,18179,1037,9.34


Interpretation: higher threshold reduces candidate volume and increases strictness.
